# AI Based Fake News Detection Tool

## Module 2 — Data Preprocessing

### CodeVedX AI/ML Internship — Project 3

---

## 📌 Purpose

This notebook performs the **complete data preprocessing pipeline**
for the Fake News Detection project. It merges the raw `Fake.csv` and
`True.csv` articles into a single, clean, machine-learning-ready
dataset by:

- Loading and inspecting both raw datasets.
- Assigning binary labels (`Fake` → `0`, `True` → `1`).
- Merging, shuffling, and resetting the index.
- Cleaning the article text with a reusable NLP pipeline.
- Tokenizing, removing stopwords, and lemmatizing.
- Engineering additional NLP features.
- Saving the processed dataset and a preprocessing report.

## 🗂️ Dataset

| File       | Description            | Rows (approx) |
|------------|------------------------|---------------|
| `Fake.csv` | Fake news articles     | ~23,481       |
| `True.csv` | Real news articles     | ~21,417       |

Both files share the columns: **`title`**, **`text`**, **`subject`**, and **`date`**.

## 🔄 Workflow

1. Import libraries and configure the environment.
2. Download required NLTK resources automatically.
3. Load and inspect `Fake.csv` and `True.csv`.
4. Assign labels and merge into one dataset.
5. Handle missing values, duplicates, and invalid text.
6. Clean text (lowercase, remove HTML/URLs/emails/punctuation/numbers, etc.).
7. Tokenize → remove stopwords → lemmatize.
8. Create `clean_text` and compute NLP features.
9. Verify, save, reload, and report.

## 🎯 Expected Output

- Merged labeled dataset with a `clean_text` column.
- NLP feature columns (`word_count`, `char_count`, `sentence_count`, `avg_word_length`).
- Processed dataset: `data/processed/fake_news_dataset.csv`.
- Preprocessing report: `outputs/reports/preprocessing_report.txt`.


## Step 2 — Import Libraries

### Why this step is required

We need a consistent set of libraries for data manipulation (`pandas`,
`numpy`), text processing (`re`, `string`, `nltk`), filesystem-safe
paths (`pathlib`), and suppressing non-critical warnings (`warnings`).

### What it does

Imports every library used throughout the notebook and verifies the
Python version.

### Expected output

A confirmation message listing the Python version and all imported
libraries.


In [1]:
# ================================================
# STEP 2: Import Libraries
# ================================================

import warnings
warnings.filterwarnings('ignore')

import re
import string
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

print(f'Python : {sys.version.split()[0]}')
print(f'Pandas : {pd.__version__}')
print(f'NumPy  : {np.__version__}')
print(f'NLTK   : {nltk.__version__}')
print('\nAll libraries imported successfully.')


Python : 3.14.0
Pandas : 2.3.3
NumPy  : 2.3.3
NLTK   : 3.9.4

All libraries imported successfully.


## Step 3 — Download Required NLTK Resources

### Why this step is required

NLTK relies on external corpora and models for tokenization
(`punkt`), stopword lists (`stopwords`), and lemmatization
(`wordnet` + `omw-1.4`). These must be present before use.

### What it does

Checks whether each resource is already available and downloads only
the missing ones, so the notebook is fully reproducible on a fresh
environment.

### Expected output

A status line for each NLTK resource (already available / downloaded).


In [2]:
# ================================================
# STEP 3: Download Required NLTK Resources
# ================================================

REQUIRED_NLTK_RESOURCES = [
    'punkt',
    'punkt_tab',
    'stopwords',
    'wordnet',
    'omw-1.4',
]

def ensure_nltk_resources(resources: list[str]) -> None:
    """Download missing NLTK resources in-place."""
    for resource in resources:
        try:
            nltk.data.find(f'tokenizers/{resource}')
        except LookupError:
            try:
                nltk.data.find(f'corpora/{resource}')
            except LookupError:
                print(f'  Downloading {resource}...')
                nltk.download(resource, quiet=True)
            else:
                print(f'  {resource}: already available')
        else:
            print(f'  {resource}: already available')


ensure_nltk_resources(REQUIRED_NLTK_RESOURCES)
print('\nAll NLTK resources ready.')


  punkt: already available
  punkt_tab: already available
  stopwords: already available



All NLTK resources ready.


## Step 4 — Load Fake.csv and True.csv

### Why this step is required

We must understand the raw data before transforming it: its size,
column names, data types, and content shape how we clean and merge.

### What it does

Defines relative paths with `pathlib.Path`, loads both CSVs with
`pandas`, and prints the shape, columns, head, tail, and `info()`
summary for each dataset.

### Expected output

Two inspection blocks — one per file — showing row/column counts,
sample rows, and non-null data-type information.


In [3]:
# ================================================
# STEP 4: Load Datasets
# ================================================

# Project-relative paths (notebooks/ -> project root)
PROJECT_ROOT = Path('..').resolve()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'

FAKE_PATH = RAW_DATA_DIR / 'Fake.csv'
TRUE_PATH = RAW_DATA_DIR / 'True.csv'

PROCESSED_CSV = PROCESSED_DATA_DIR / 'fake_news_dataset.csv'
REPORT_TXT = REPORTS_DIR / 'preprocessing_report.txt'

# Ensure output directories exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Fake.csv     : {FAKE_PATH}')
print(f'True.csv     : {TRUE_PATH}')


Project root : D:\New Project\CodeVedex Projects\CODEVEDX\Project-03-Fake-News-Detection
Fake.csv     : D:\New Project\CodeVedex Projects\CODEVEDX\Project-03-Fake-News-Detection\data\raw\Fake.csv
True.csv     : D:\New Project\CodeVedex Projects\CODEVEDX\Project-03-Fake-News-Detection\data\raw\True.csv


In [4]:
# ================================================
# STEP 4b: Inspect Fake.csv
# ================================================

df_fake = pd.read_csv(FAKE_PATH)

print('=' * 70)
print('FAKE NEWS DATASET (Fake.csv)')
print('=' * 70)
print(f'Shape: {df_fake.shape[0]:,} rows x {df_fake.shape[1]} columns')
print(f'Columns: {list(df_fake.columns)}')

print('\n--- Head (first 5 rows) ---')
df_fake.head()


FAKE NEWS DATASET (Fake.csv)
Shape: 23,481 rows x 4 columns
Columns: ['title', 'text', 'subject', 'date']

--- Head (first 5 rows) ---


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
# ================================================
# STEP 4c: Inspect Fake.csv (tail + info)
# ================================================

print('--- Tail (last 5 rows) ---')
df_fake.tail()


--- Tail (last 5 rows) ---


,title,text,subject,date
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"
23480,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016"


In [6]:
# ================================================
# STEP 4d: Fake.csv info
# ================================================

df_fake.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [7]:
# ================================================
# STEP 4e: Inspect True.csv
# ================================================

df_true = pd.read_csv(TRUE_PATH)

print('=' * 70)
print('TRUE NEWS DATASET (True.csv)')
print('=' * 70)
print(f'Shape: {df_true.shape[0]:,} rows x {df_true.shape[1]} columns')
print(f'Columns: {list(df_true.columns)}')

print('\n--- Head (first 5 rows) ---')
df_true.head()


TRUE NEWS DATASET (True.csv)
Shape: 21,417 rows x 4 columns
Columns: ['title', 'text', 'subject', 'date']

--- Head (first 5 rows) ---


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [8]:
# ================================================
# STEP 4f: Inspect True.csv (tail + info)
# ================================================

print('--- Tail (last 5 rows) ---')
df_true.tail()


--- Tail (last 5 rows) ---


,title,text,subject,date
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017"
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017"
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017"
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017"
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017"


In [9]:
# ================================================
# STEP 4g: True.csv info
# ================================================

df_true.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


## Step 5 — Add Labels

### Why this step is required

Supervised ML needs a target variable. We encode the article type as a
binary label so a classifier can learn to distinguish fake from true
news.

### What it does

Creates a `label` column in each DataFrame: `Fake.csv` → `0`,
`True.csv` → `1`.

### Expected output

A `label` column appended to each raw DataFrame.


In [10]:
# ================================================
# STEP 5: Add Labels
# ================================================

# Fake news -> 0, True news -> 1
df_fake['label'] = 0
df_true['label'] = 1

print('Label added to Fake.csv  -> label = 0 (Fake)')
print('Label added to True.csv  -> label = 1 (True)')

print(f'\nFake.csv label counts:',
      f'  {df_fake["label"].value_counts().to_dict()}')
print(f'True.csv label counts:',
      f'  {df_true["label"].value_counts().to_dict()}')


Label added to Fake.csv  -> label = 0 (Fake)
Label added to True.csv  -> label = 1 (True)

Fake.csv label counts:   {0: 23481}
True.csv label counts:   {1: 21417}


## Step 6 — Merge, Shuffle, and Reset Index

### Why this step is required

Combining both files creates the full training corpus. Shuffling
removes the ordering bias (all fake rows first), and resetting the
index produces a clean sequential index after the shuffle.

### What it does

Concatenates the two labeled DataFrames, shuffles with a fixed random
seed for reproducibility, and resets the index.

### Expected output

A single merged DataFrame with roughly `44,898` rows and a fresh
sequential index.


In [11]:
# ================================================
# STEP 6: Merge, Shuffle, Reset Index
# ================================================

df = pd.concat([df_fake, df_true], axis=0, ignore_index=True)

# Shuffle with fixed seed for reproducibility
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f'Merged dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Shuffled and index reset.')


Merged dataset: 44,898 rows x 5 columns
Shuffled and index reset.


## Step 7 — Merged Shape and Class Distribution

### Why this step is required

Before cleaning we verify the merged size and check whether the two
classes are balanced. Class balance affects how we interpret accuracy
later and whether any rebalancing is needed.

### What it does

Prints the merged shape and the count/percentage of each label.

### Expected output

The merged shape and a label distribution table (counts + percentages).


In [12]:
# ================================================
# STEP 7: Merged Shape & Class Distribution
# ================================================

print(f'Merged shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

print('\nClass distribution:')
class_dist = df['label'].value_counts().sort_index()
class_pct = df['label'].value_counts(normalize=True).sort_index() * 100

dist_df = pd.DataFrame({
    'Count': class_dist,
    'Percentage': class_pct.round(2),
})
dist_df.index = ['Fake (0)', 'True (1)']
dist_df.index.name = 'Label'
display(dist_df)


Merged shape: 44,898 rows x 5 columns

Class distribution:


,Count,Percentage
Label,,
Fake (0),23481,52.3
True (1),21417,47.7


## Step 8 — Missing Values, Duplicates, Empty Text, Empty Title

### Why this step is required

Raw text data often contains missing cells, fully duplicated rows, or
blank `title`/`text` fields. Detecting these before cleaning avoids
silent data loss and downstream errors.

### What it does

Reports the total missing values, count of duplicate rows, and counts
of empty/whitespace-only `text` and `title` entries.

### Expected output

A summary of data-quality issues in the merged dataset.


In [13]:
# ================================================
# STEP 8: Quality Checks (missing/duplicates/empty)
# ================================================

def is_blank(series: pd.Series) -> pd.Series:
    """Return True where the series is null or whitespace-only."""
    return series.isna() | series.astype(str).str.strip().eq('')


print('=' * 70)
print('DATA QUALITY CHECK (BEFORE CLEANING)')
print('=' * 70)

missing_total = int(df.isna().sum().sum())
duplicates_total = int(df.duplicated().sum())
empty_text = int(is_blank(df['text']).sum())
empty_title = int(is_blank(df['title']).sum())

print(f'  Missing values (total)          : {missing_total:,}')
print(f'  Duplicate rows                  : {duplicates_total:,}')
print(f'  Empty/blank text                : {empty_text:,}')
print(f'  Empty/blank title               : {empty_title:,}')

print('\nMissing values per column:')
print(df.isna().sum().to_string())


DATA QUALITY CHECK (BEFORE CLEANING)


  Missing values (total)          : 0
  Duplicate rows                  : 209
  Empty/blank text                : 631
  Empty/blank title               : 0

Missing values per column:
title      0
text       0
subject    0
date       0
label      0


## Step 9 — Handle Missing Values, Duplicates, and Invalid Text

### Why this step is required

Machine learning models cannot consume missing or duplicated rows, and
blank text produces meaningless cleaned documents. Removing or fixing
these issues yields a clean, model-ready dataset.

### What it does

- Converts all text fields to strings and strips surrounding whitespace.
- Drops rows with blank `title` or blank `text`.
- Drops fully duplicated rows (keeping the first occurrence).
- Reports exactly how many rows were removed at each stage.

### Expected output

A cleaned merged DataFrame with no missing text, no blank titles/text,
and no exact duplicate rows.


In [14]:
# ================================================
# STEP 9: Handle Missing / Duplicates / Invalid Text
# ================================================

before_rows = len(df)

# 1. Coerce text columns to string and strip whitespace
for col in ['title', 'text', 'subject', 'date']:
    df[col] = df[col].astype(str).str.strip()

# 2. Drop rows with blank title or blank text
df = df[~is_blank(df['title'])].copy()
df = df[~is_blank(df['text'])].copy()

# 3. Drop exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

after_rows = len(df)
print(f'Rows before cleaning : {before_rows:,}')
print(f'Rows after cleaning  : {after_rows:,}')
print(f'Rows removed         : {before_rows - after_rows:,}')

print(f'\nRemaining missing values: {int(df.isna().sum().sum()):,}')
print(f'Remaining duplicates    : {int(df.duplicated().sum()):,}')


Rows before cleaning : 44,898
Rows after cleaning  : 44,056
Rows removed         : 842

Remaining missing values: 0


Remaining duplicates    : 0


## Step 10 — Complete NLP Cleaning

### Why this step is required

Raw news text is noisy: it contains HTML, URLs, email addresses,
punctuation, numbers, special characters, and non-ASCII symbols. These
add noise to vector representations and rarely help classification.
Normalizing the text reduces the vocabulary size and improves model
performance.

### What it does

Defines a reusable `clean_text()` function that applies, in order:

1. Lowercasing.
2. Removing HTML tags.
3. Removing URLs.
4. Removing email addresses.
5. Removing punctuation.
6. Removing numbers.
7. Removing special characters.
8. Removing non-ASCII characters.
9. Removing extra whitespace.

### Expected output

A clean, normalized text string for every article.


In [15]:
# ================================================
# STEP 10: Complete NLP Cleaning
# ================================================

# Regex patterns (compiled once for speed)
_HTML_TAG_RE = re.compile(r'<[^>]+>')
_URL_RE = re.compile(r'http\S+|www\.\S+')
_EMAIL_RE = re.compile(r'\S+@\S+\.\S+')
_PUNCT_RE = re.compile(r'[' + re.escape(string.punctuation) + r']')
_DIGIT_RE = re.compile(r'\d+')
_SPECIAL_RE = re.compile(r'[^a-zA-Z\s]')
_MULTI_SPACE_RE = re.compile(r'\s+')
_NON_ASCII_RE = re.compile(r'[^\x00-\x7F]+')


def clean_text(text: str) -> str:
    """Normalize raw article text into clean, lowercase ASCII text."""
    if not isinstance(text, str):
        return ''

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove HTML tags
    text = _HTML_TAG_RE.sub(' ', text)

    # 3. Remove URLs
    text = _URL_RE.sub(' ', text)

    # 4. Remove email addresses
    text = _EMAIL_RE.sub(' ', text)

    # 5. Remove punctuation
    text = _PUNCT_RE.sub(' ', text)

    # 6. Remove numbers
    text = _DIGIT_RE.sub(' ', text)

    # 7. Remove special characters (anything not a letter/space)
    text = _SPECIAL_RE.sub(' ', text)

    # 8. Remove non-ASCII characters
    text = _NON_ASCII_RE.sub(' ', text)

    # 9. Remove extra whitespace
    text = _MULTI_SPACE_RE.sub(' ', text).strip()

    return text


# Quick sanity check
sample = '<p>Breaking NEWS! Read http://example.com today\n' \
         'email me at foo@bar.com 12345 #MAGA!!!</p>'
print('Before:', sample)
print('After :', clean_text(sample))


Before: <p>Breaking NEWS! Read http://example.com today
email me at foo@bar.com 12345 #MAGA!!!</p>
After : breaking news read today email me at maga


## Step 11 — Tokenization

### Why this step is required

Tokenization splits text into individual words (tokens), the atomic
units used for stopword removal, lemmatization, and later TF-IDF
vectorization.

### What it does

Uses NLTK's `word_tokenize` to convert each cleaned article into a
list of word tokens.

### Expected output

A list of word tokens for each article.


In [16]:
# ================================================
# STEP 11: Tokenization
# ================================================

def tokenize(text: str) -> list[str]:
    """Tokenize cleaned text into a list of word tokens."""
    if not text or not isinstance(text, str):
        return []
    return word_tokenize(text)


tokens_sample = tokenize(clean_text(df.loc[0, 'text']))
print(f'Sample tokens ({len(tokens_sample)}):')
print(tokens_sample[:25])


Sample tokens (172):
['st', 'century', 'wire', 'says', 'ben', 'stein', 'reputable', 'professor', 'from', 'pepperdine', 'university', 'also', 'of', 'some', 'hollywood', 'fame', 'appearing', 'in', 'tv', 'shows', 'and', 'films', 'such', 'as', 'ferris']


## Step 12 — Stopword Removal

### Why this step is required

High-frequency stopwords (e.g., "the", "and", "is") carry little
discriminative signal and inflate the vocabulary. Removing them
focuses the model on meaningful content words.

**Important:** we keep the negation words `not`, `no`, and `never`
because they strongly affect sentiment and meaning in news text.

### What it does

Builds an English stopword set, removes the protected negation words,
and filters tokens to exclude remaining stopwords and single-character
tokens.

### Expected output

A token list with stopwords removed but negation words preserved.


In [17]:
# ================================================
# STEP 12: Stopword Removal
# ================================================

# Negation words that must be preserved
NEGATION_WORDS = {'not', 'no', 'never'}

# Build the stopword set once
_ENGLISH_STOPWORDS = set(stopwords.words('english')) - NEGATION_WORDS


def remove_stopwords(tokens: list[str]) -> list[str]:
    """Remove stopwords while keeping negation words and short words."""
    return [
        token for token in tokens
        if token not in _ENGLISH_STOPWORDS and len(token) > 1
    ]


print(f'English stopwords loaded (excluding negations): {len(_ENGLISH_STOPWORDS):,}')
print(f'Protected negation words: {sorted(NEGATION_WORDS)}')


English stopwords loaded (excluding negations): 196
Protected negation words: ['never', 'no', 'not']


## Step 13 — Lemmatization

### Why this step is required

Lemmatization reduces words to their dictionary base form (e.g.,
“running” → “run”). This collapses inflectional variants into one
feature, shrinking the vocabulary and improving generalization.

### What it does

Uses NLTK's `WordNetLemmatizer` to lemmatize each token after
stopword removal.

### Expected output

A list of lemmatized tokens ready to be joined into `clean_text`.


In [18]:
# ================================================
# STEP 13: Lemmatization
# ================================================

_LEMMATIZER = WordNetLemmatizer()


def lemmatize(tokens: list[str]) -> list[str]:
    """Lemmatize a list of word tokens to their base forms."""
    return [_LEMMATIZER.lemmatize(token) for token in tokens]


# Demonstrate lemmatization on a tiny example
demo_tokens = ['running', 'cars', 'better', 'went', 'studies']
print('Before:', demo_tokens)
print('After :', lemmatize(demo_tokens))


Before: ['running', 'cars', 'better', 'went', 'studies']


After : ['running', 'car', 'better', 'went', 'study']


## Step 14 — Create the `clean_text` Column

### Why this step is required

The pipeline (clean → tokenize → remove stopwords → lemmatize) must be
applied to every article. Storing the result in a dedicated
`clean_text` column keeps the original `title`/`text` untouched for
inspection and reproducibility.

### What it does

Applies the full preprocessing pipeline to each row's `text` column
and stores the joined, lemmatized output in `clean_text`. Rows whose
cleaned text becomes empty (no meaningful content remained after
cleaning) are removed, because they carry no signal for the model.

### Expected output

A `clean_text` column containing the preprocessed article text, with
empty cleaned documents removed.


In [19]:
# ================================================
# STEP 14: Create clean_text Column
# ================================================

def preprocess_article(text: str) -> str:
    """Full pipeline: clean -> tokenize -> remove stopwords -> lemmatize."""
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    return ' '.join(tokens)


df['clean_text'] = df['text'].apply(preprocess_article)

rows_before_clean = len(df)

# Drop rows whose cleaned text is empty (no meaningful content)
df = df[~is_blank(df['clean_text'])].copy().reset_index(drop=True)

rows_removed_empty = rows_before_clean - len(df)
print(f'clean_text created for {rows_before_clean:,} rows.')
print(f'Removed {rows_removed_empty:,} rows with empty clean_text.')
df[['title', 'text', 'clean_text']].head(3)


clean_text created for 44,056 rows.
Removed 85 rows with empty clean_text.


,title,text,clean_text
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",st century wire say ben stein reputable profes...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,washington reuters president donald trump remo...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,reuters puerto rico governor ricardo rossello ...


## Step 15 — Compute Additional NLP Features

### Why this step is required

Beyond raw text, numeric features (word/character/sentence counts and
average word length) give models useful signals about article length
and density, and can help detect anomalies.

### What it does

Adds four engineered columns computed from the cleaned text:

| Column                | Description                    |
|-----------------------|--------------------------------|
| `word_count`          | Number of words                |
| `char_count`          | Number of characters           |
| `sentence_count`      | Number of sentences            |
| `avg_word_length`     | Mean characters per word       |

### Expected output

Four new numeric columns plus a preview of the enhanced dataset.


In [20]:
# ================================================
# STEP 15: Compute Additional NLP Features
# ================================================

def compute_word_count(text: str) -> int:
    """Count words in a text string."""
    return len(str(text).split())


def compute_char_count(text: str) -> int:
    """Count characters in a text string."""
    return len(str(text))


def compute_sentence_count(text: str) -> int:
    """Count sentences using NLTK's sentence tokenizer."""
    return len(sent_tokenize(str(text)))


def compute_avg_word_length(text: str) -> float:
    """Average word length (chars per word), 0 if no words."""
    words = str(text).split()
    if not words:
        return 0.0
    return round(sum(len(word) for word in words) / len(words), 2)


df['word_count'] = df['clean_text'].apply(compute_word_count)
df['char_count'] = df['clean_text'].apply(compute_char_count)
df['sentence_count'] = df['clean_text'].apply(compute_sentence_count)
df['avg_word_length'] = df['clean_text'].apply(compute_avg_word_length)

print('NLP features added: word_count, char_count, sentence_count, avg_word_length')
df[['clean_text', 'word_count', 'char_count', 'sentence_count', 'avg_word_length']].head(5)


NLP features added: word_count, char_count, sentence_count, avg_word_length


,clean_text,word_count,char_count,sentence_count,avg_word_length
0,st century wire say ben stein reputable profes...,102,731,1,6.18
1,washington reuters president donald trump remo...,466,3475,1,6.46
2,reuters puerto rico governor ricardo rossello ...,176,1257,1,6.15
3,monday donald trump embarrassed country accide...,110,854,1,6.77
4,glasgow scotland reuters presidential candidat...,300,2189,1,6.30


## Step 16 — Verify Preprocessing (Original → Clean)

### Why this step is required

We must visually confirm the pipeline works on real articles before
saving. Spot-checking several examples shows how raw text is
transformed and catches any unexpected behaviour.

### What it does

Prints a side-by-side comparison of the original `text` and the
processed `clean_text` for multiple sample articles.

### Expected output

A readable original → clean comparison table for several rows.


In [21]:
# ================================================
# STEP 16: Verify Preprocessing
# ================================================

examples = df.sample(3, random_state=7)

for idx, row in examples.iterrows():
    print('=' * 80)
    print(f"Row {idx}  |  Label: {'Fake' if row['label'] == 0 else 'True'}")
    print('-' * 80)
    print('ORIGINAL TEXT:')
    print(str(row['text'])[:500])
    print('\nCLEAN TEXT:')
    print(str(row['clean_text'])[:500])
    print()


Row 34829  |  Label: Fake
--------------------------------------------------------------------------------
ORIGINAL TEXT:
Donald Trump once bragged that he had  the best words  but researchers found that his speeches were at an average of about a 7th-grade reading level. Trump generally scored the lowest of all of his rivals. But, Trump being Trump, he s doing it again and bragging about his  beautiful flowing sentences. This week, Trump canceled a much-anticipated press conference on his private business dealings, saying he was too busy nominating Cabinet members, but he was seen meeting with Kanye West. Trump pro

CLEAN TEXT:
donald trump bragged best word researcher found speech average th grade reading level trump generally scored lowest rival trump trump bragging beautiful flowing sentence week trump canceled much anticipated press conference private business dealing saying busy nominating cabinet member seen meeting kanye west trump promised address conflict interest suppose meet

## Step 17 — Quality Checks

### Why this step is required

Before exporting we assert the dataset meets minimum quality gates:
no missing values, no duplicates, no empty cleaned text, and a
balanced label distribution. This prevents silently shipping a
corrupt dataset to later modules.

### What it does

Runs automated assertions and prints a summary of each quality gate.

### Expected output

All checks pass and a confirmation summary is printed.


In [22]:
# ================================================
# STEP 17: Quality Checks
# ================================================

print('=' * 70)
print('FINAL QUALITY CHECKS')
print('=' * 70)

checks = {
    'No missing values': int(df.isna().sum().sum()) == 0,
    'No duplicate rows': int(df.duplicated().sum()) == 0,
    'No empty clean_text': int(is_blank(df['clean_text']).sum()) == 0,
    'Balanced labels': df['label'].value_counts().min() > 0.4 * len(df),
}

for name, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {name}')

assert all(checks.values()), 'One or more quality checks failed!'
print('\n✅ All quality checks passed — dataset is model-ready.')


FINAL QUALITY CHECKS


  [PASS] No missing values
  [PASS] No duplicate rows
  [PASS] No empty clean_text
  [PASS] Balanced labels

✅ All quality checks passed — dataset is model-ready.


## Step 18 — Save the Processed Dataset

### Why this step is required

Persisting the cleaned dataset to CSV makes it available to later
modules (EDA, model training) without re-running the entire
preprocessing pipeline.

### What it does

Saves the processed DataFrame to `data/processed/fake_news_dataset.csv`
using `utf-8` encoding and `index=False`.

### Expected output

A CSV file written to `data/processed/fake_news_dataset.csv`.


In [23]:
# ================================================
# STEP 18: Save Processed Dataset
# ================================================

df.to_csv(PROCESSED_CSV, index=False, encoding='utf-8')

print(f'Processed dataset saved to: {PROCESSED_CSV}')
print(f'Rows x Columns: {df.shape[0]:,} x {df.shape[1]}')


Processed dataset saved to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-03-Fake-News-Detection\data\processed\fake_news_dataset.csv
Rows x Columns: 43,971 x 10


## Step 19 — Reload and Verify the Saved Dataset

### Why this step is required

Reloading the CSV from disk verifies the file is well-formed and
matches the in-memory DataFrame — a critical reproducibility check.

### What it does

Reads the saved CSV back with `pandas`, then prints its shape, column
names, and data types.

### Expected output

A confirmation that the reloaded dataset matches expectations.


In [24]:
# ================================================
# STEP 19: Reload and Verify
# ================================================

df_reloaded = pd.read_csv(PROCESSED_CSV)

print('Reloaded dataset from disk:')
print(f'Shape       : {df_reloaded.shape[0]:,} rows x {df_reloaded.shape[1]} columns')
print(f'Columns     : {list(df_reloaded.columns)}')
print(f'Data types  :')
print(df_reloaded.dtypes.to_string())

assert df_reloaded.shape == df.shape, 'Reloaded shape mismatch!'
print('\n✅ Reloaded dataset matches the in-memory DataFrame.')


Reloaded dataset from disk:
Shape       : 43,971 rows x 10 columns
Columns     : ['title', 'text', 'subject', 'date', 'label', 'clean_text', 'word_count', 'char_count', 'sentence_count', 'avg_word_length']
Data types  :
title               object
text                object
subject             object
date                object
label                int64
clean_text          object
word_count           int64
char_count           int64
sentence_count       int64
avg_word_length    float64

✅ Reloaded dataset matches the in-memory DataFrame.


## Step 20 — Generate the Preprocessing Report

### Why this step is required

A concise text report documents the data cleaning summary for
stakeholders and later modules: how many rows were removed, the final
vocabulary size, average text length, and label distribution.

### What it does

Computes key metrics from the raw and cleaned DataFrames and writes
them to `outputs/reports/preprocessing_report.txt`.

### Expected output

A human-readable `.txt` report saved to `outputs/reports/`.


In [25]:
# ================================================
# STEP 20: Generate Preprocessing Report
# ================================================

def build_vocabulary(texts: pd.Series) -> set[str]:
    """Build a set of all unique words across the given texts."""
    vocab: set[str] = set()
    for text in texts.astype(str):
        vocab.update(str(text).split())
    return vocab


original_rows = len(df_fake) + len(df_true)
final_rows = len(df)
duplicates_removed = duplicates_total
missing_removed = missing_total
vocab_size = len(build_vocabulary(df['clean_text']))
avg_text_length = round(df['clean_text'].str.len().mean(), 2)

label_dist = df['label'].value_counts().sort_index().to_dict()

report_lines = [
    '=' * 60,
    'AI BASED FAKE NEWS DETECTION TOOL',
    'MODULE 2 — DATA PREPROCESSING REPORT',
    '=' * 60,
    '',
    f'Original rows (Fake + True) : {original_rows:,}',
    f'Final rows (after cleaning) : {final_rows:,}',
    f'Rows removed                : {original_rows - final_rows:,}',
    f'Duplicates removed          : {duplicates_removed:,}',
    f'Missing values removed      : {missing_removed:,}',
    f'Vocabulary size             : {vocab_size:,}',
    f'Average text length (chars) : {avg_text_length}',
    '',
    'Label distribution:',
    f'  Fake (0) : {label_dist.get(0, 0):,}',
    f'  True (1) : {label_dist.get(1, 0):,}',
    '',
    f'Processed dataset : {PROCESSED_CSV}',
    f'Report generated  : data_preprocessing.ipynb (Module 2)',
    '=' * 60,
]

REPORT_TXT.write_text('\n'.join(report_lines), encoding='utf-8')

print('\n'.join(report_lines))
print(f'\n✅ Report saved to: {REPORT_TXT}')


AI BASED FAKE NEWS DETECTION TOOL
MODULE 2 — DATA PREPROCESSING REPORT

Original rows (Fake + True) : 44,898
Final rows (after cleaning) : 43,971
Rows removed                : 927
Duplicates removed          : 209
Missing values removed      : 0
Vocabulary size             : 102,713
Average text length (chars) : 1723.66

Label distribution:
  Fake (0) : 22,761
  True (1) : 21,210

Processed dataset : D:\New Project\CodeVedex Projects\CODEVEDX\Project-03-Fake-News-Detection\data\processed\fake_news_dataset.csv
Report generated  : data_preprocessing.ipynb (Module 2)

✅ Report saved to: D:\New Project\CodeVedex Projects\CODEVEDX\Project-03-Fake-News-Detection\outputs\reports\preprocessing_report.txt
